# CCNF vs UNet Reconstruction Diagnostics

This notebook isolates whether reconstruction blurriness in the CCNF (neural field decoder)
originates in **per-patch model output** (spectral bias) or the **reassembly pipeline** (stitching/scaling).

## Hypothesis
The MLP-based neural field decoder suffers from spectral bias — it learns low-frequency
solutions that minimize Poisson loss but lack high-frequency spatial detail. The UNet preserves
spatial frequencies via skip connections, producing sharper patches despite compressing phase range.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import copy
import os

from ptycho_torch.config_params import (
    DataConfig, ModelConfig, TrainingConfig, InferenceConfig,
    update_existing_config,
)
from ptycho_torch.model import PtychoPINN_Lightning
from ptycho_torch.beta_modules.inference_indexed import (
    load_model_from_checkpoint, build_indexed_dataset,
)
from ptycho_torch.reassembly import (
    reconstruct_image_barycentric,
    VarProScaler,
    VectorizedWeightedAccumulator,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 4)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Configuration
Set paths to your checkpoints and data directory below.

In [ ]:
# ========== USER CONFIGURATION ==========

# CCNF checkpoint (Lightning .ckpt path or run directory)
CCNF_CKPT = '/path/to/ccnf/checkpoints/last.ckpt'

# UNet checkpoint — choose ONE method:
#   Option A: Lightning .ckpt path
UNET_CKPT = '/path/to/unet/checkpoints/best-checkpoint.ckpt'
UNET_LOAD_METHOD = 'lightning'  # 'lightning' or 'mlflow'

#   Option B: MLFlow run ID (set UNET_LOAD_METHOD = 'mlflow')
UNET_MLFLOW_RUN_ID = ''
MLFLOW_PATH = '../mlruns'

# Data directory (containing .npz scan files)
PTYCHO_DIR = '/path/to/data/'

# Inference parameters
BATCH_SIZE = 4
MIDDLE_TRIM = 50
EXPERIMENT_NUMBER = 0
N_VIS_PATCHES = 4  # number of patches to visualize

## 2. Load Models

In [ ]:
# --- CCNF ---
ccnf_model, ccnf_dc, ccnf_mc, ccnf_tc, ccnf_ic = load_model_from_checkpoint(
    CCNF_CKPT, device=DEVICE
)
ccnf_model.eval()
print(f'CCNF architecture: {getattr(ccnf_mc, "architecture", "unet")}')
print(f'CCNF params: {sum(p.numel() for p in ccnf_model.parameters()):,}')

# --- UNet ---
if UNET_LOAD_METHOD == 'lightning':
    from ptycho_torch.lightning_utils import load_checkpoint_with_configs
    unet_lightning, unet_configs = load_checkpoint_with_configs(
        UNET_CKPT, PtychoPINN_Lightning, device=DEVICE
    )
    unet_model = unet_lightning
    unet_dc, unet_mc, unet_tc, unet_ic, _ = unet_configs
else:
    import mlflow
    from ptycho_torch.utils import load_all_configs_from_mlflow
    tracking_uri = f'file:{os.path.abspath(MLFLOW_PATH)}'
    mlflow.set_tracking_uri(tracking_uri)
    unet_dc, unet_mc, unet_tc, unet_ic, _ = load_all_configs_from_mlflow(
        UNET_MLFLOW_RUN_ID, tracking_uri
    )
    unet_model = mlflow.pytorch.load_model(f'runs:/{UNET_MLFLOW_RUN_ID}/model')
    unet_model.to(DEVICE)

unet_model.eval()
print(f'UNet architecture: {getattr(unet_mc, "architecture", "unet")}')
print(f'UNet params: {sum(p.numel() for p in unet_model.parameters()):,}')

## 3. Build Dataset & DataLoader

In [ ]:
from ptycho_torch.beta_modules.dataloader_index import PtychoDatasetIndexed
from ptycho_torch.dataloader import PtychoDataset, Collate_Lightning
from torch.utils.data import DataLoader

# Use CCNF configs for dataset (both models should work on same data)
dc = copy.deepcopy(ccnf_dc)
tc = copy.deepcopy(ccnf_tc)
update_existing_config(dc, {
    'probe_normalize': False,
    'x_bounds': [0.03, 0.97],
    'y_bounds': [0.03, 0.97],
})

ic = copy.deepcopy(ccnf_ic)
ic.batch_size = BATCH_SIZE
ic.middle_trim = MIDDLE_TRIM
ic.experiment_number = EXPERIMENT_NUMBER

# Build indexed dataset
mmap_dir = os.path.join(os.path.dirname(PTYCHO_DIR.rstrip('/')), 'memmap_diag')
dataset = build_indexed_dataset(
    PTYCHO_DIR, ccnf_mc, dc, tc, mmap_dir=mmap_dir,
)

# Get the experiment subset
subset = dataset.get_experiment_dataset(EXPERIMENT_NUMBER)

collate_fn = Collate_Lightning(dc, ccnf_mc)
loader = DataLoader(
    subset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
)

# Grab first batch
batch = next(iter(loader))
print(f'Batch images shape: {batch[0]["images"].shape}')
print(f'Batch coords_relative shape: {batch[0]["coords_relative"].shape}')
print(f'Probe shape: {batch[1].shape}')

## 4. Verify Output Format

In [ ]:
x = batch[0]['images'].to(DEVICE)
positions = batch[0]['coords_relative'].to(DEVICE)
probe = batch[1].to(DEVICE)
in_scale = batch[0]['rms_scaling_constant'].to(DEVICE)

with torch.no_grad():
    ccnf_out = ccnf_model.forward_predict(x, positions, probe, in_scale)
    unet_out = unet_model.forward_predict(x, positions, probe, in_scale)

print('=== CCNF Output ===')
print(f'  dtype: {ccnf_out.dtype}, shape: {ccnf_out.shape}')
print(f'  real range: [{ccnf_out.real.min():.4f}, {ccnf_out.real.max():.4f}]')
print(f'  imag range: [{ccnf_out.imag.min():.4f}, {ccnf_out.imag.max():.4f}]')
print(f'  |O| range:  [{ccnf_out.abs().min():.4f}, {ccnf_out.abs().max():.4f}]')

print('\n=== UNet Output ===')
print(f'  dtype: {unet_out.dtype}, shape: {unet_out.shape}')
print(f'  real range: [{unet_out.real.min():.4f}, {unet_out.real.max():.4f}]')
print(f'  imag range: [{unet_out.imag.min():.4f}, {unet_out.imag.max():.4f}]')
print(f'  |O| range:  [{unet_out.abs().min():.4f}, {unet_out.abs().max():.4f}]')

## 5. Individual Patch Visualization

Side-by-side comparison of CCNF vs UNet per-patch predictions.
If CCNF patches are individually blurry, the issue is **spectral bias in the decoder**.
If patches are sharp but the assembled image is blurry, the issue is **reassembly**.

In [ ]:
n_show = min(N_VIS_PATCHES, x.shape[0])

# Use channel 0 for visualization
c_idx = 0

fig, axes = plt.subplots(5, n_show * 2, figsize=(4 * n_show * 2, 16))
if n_show * 2 == 1:
    axes = axes[:, None]

row_labels = [
    'Input diffraction (log)',
    'Predicted Real',
    'Predicted Imag',
    'Predicted |O|',
    'Predicted phase',
]

for p in range(n_show):
    col_ccnf = p * 2
    col_unet = p * 2 + 1

    # Input diffraction
    diff_patch = x[p, c_idx].cpu().numpy()
    for col in [col_ccnf, col_unet]:
        axes[0, col].imshow(np.log1p(diff_patch), cmap='viridis')
        axes[0, col].set_title(f'Patch {p} - {"CCNF" if col == col_ccnf else "UNet"}')

    # CCNF predictions
    cr = ccnf_out[p, c_idx].cpu()
    axes[1, col_ccnf].imshow(cr.real.numpy(), cmap='RdBu_r'); axes[1, col_ccnf].set_title('CCNF Real')
    axes[2, col_ccnf].imshow(cr.imag.numpy(), cmap='RdBu_r'); axes[2, col_ccnf].set_title('CCNF Imag')
    axes[3, col_ccnf].imshow(cr.abs().numpy(), cmap='gray'); axes[3, col_ccnf].set_title('CCNF |O|')
    axes[4, col_ccnf].imshow(cr.angle().numpy(), cmap='twilight'); axes[4, col_ccnf].set_title('CCNF phase')

    # UNet predictions
    ur = unet_out[p, c_idx].cpu()
    axes[1, col_unet].imshow(ur.real.numpy(), cmap='RdBu_r'); axes[1, col_unet].set_title('UNet Real')
    axes[2, col_unet].imshow(ur.imag.numpy(), cmap='RdBu_r'); axes[2, col_unet].set_title('UNet Imag')
    axes[3, col_unet].imshow(ur.abs().numpy(), cmap='gray'); axes[3, col_unet].set_title('UNet |O|')
    axes[4, col_unet].imshow(ur.angle().numpy(), cmap='twilight'); axes[4, col_unet].set_title('UNet phase')

for i, lbl in enumerate(row_labels):
    axes[i, 0].set_ylabel(lbl, fontsize=10)

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('patch_comparison.png', bbox_inches='tight')
plt.show()

## 6. Spatial Frequency Analysis

Radially-averaged power spectra quantify spectral bias.
If CCNF has less high-frequency power, the MLP decoder is the bottleneck.

In [ ]:
def radial_power_spectrum(img_2d):
    """Compute radially averaged power spectrum of a 2D image."""
    f2 = np.fft.fftshift(np.fft.fft2(img_2d))
    power = np.abs(f2) ** 2
    H, W = power.shape
    cy, cx = H // 2, W // 2
    Y, X = np.ogrid[:H, :W]
    r = np.sqrt((X - cx)**2 + (Y - cy)**2).astype(int)
    max_r = min(cy, cx)
    radial_mean = np.zeros(max_r)
    for ri in range(max_r):
        mask = r == ri
        if mask.any():
            radial_mean[ri] = power[mask].mean()
    freqs = np.arange(max_r) / max_r * 0.5  # normalized frequency
    return freqs, radial_mean


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for p in range(min(n_show, 4)):
    cr = ccnf_out[p, c_idx].cpu()
    ur = unet_out[p, c_idx].cpu()

    # Real channel power spectrum
    f_c, ps_c = radial_power_spectrum(cr.real.numpy())
    f_u, ps_u = radial_power_spectrum(ur.real.numpy())
    axes[0].semilogy(f_c, ps_c, color=f'C{p}', linestyle='-', alpha=0.7, label=f'CCNF p{p}')
    axes[0].semilogy(f_u, ps_u, color=f'C{p}', linestyle='--', alpha=0.7, label=f'UNet p{p}')

    # Imag channel power spectrum
    f_c, ps_c = radial_power_spectrum(cr.imag.numpy())
    f_u, ps_u = radial_power_spectrum(ur.imag.numpy())
    axes[1].semilogy(f_c, ps_c, color=f'C{p}', linestyle='-', alpha=0.7)
    axes[1].semilogy(f_u, ps_u, color=f'C{p}', linestyle='--', alpha=0.7)

axes[0].set_title('Real Channel — Radial Power Spectrum')
axes[0].set_xlabel('Normalized frequency')
axes[0].set_ylabel('Power')
axes[0].legend(fontsize=7, ncol=2)
axes[1].set_title('Imag Channel — Radial Power Spectrum')
axes[1].set_xlabel('Normalized frequency')

plt.tight_layout()
plt.savefig('power_spectrum_comparison.png', bbox_inches='tight')
plt.show()

## 7. Value Distribution Comparison

Check whether output values are saturated at activation bounds.
- CCNF real: expected range [-0.8, 1.2] (from `0.2 + tanh`)
- CCNF imag: expected range [-1.2, 1.2] (from `1.2 * tanh`)
- UNet: expected range [-1, 1] (from `tanh`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ccnf_real_vals = ccnf_out[:, c_idx].real.cpu().numpy().ravel()
ccnf_imag_vals = ccnf_out[:, c_idx].imag.cpu().numpy().ravel()
unet_real_vals = unet_out[:, c_idx].real.cpu().numpy().ravel()
unet_imag_vals = unet_out[:, c_idx].imag.cpu().numpy().ravel()

axes[0].hist(ccnf_real_vals, bins=100, alpha=0.6, label='CCNF real', color='C0')
axes[0].hist(unet_real_vals, bins=100, alpha=0.6, label='UNet real', color='C1')
axes[0].axvline(-0.8, color='C0', ls=':', label='CCNF min (-0.8)')
axes[0].axvline(1.2, color='C0', ls=':')
axes[0].axvline(-1.0, color='C1', ls=':', label='UNet min (-1.0)')
axes[0].axvline(1.0, color='C1', ls=':')
axes[0].set_title('Real Channel Distribution')
axes[0].legend(fontsize=8)

axes[1].hist(ccnf_imag_vals, bins=100, alpha=0.6, label='CCNF imag', color='C0')
axes[1].hist(unet_imag_vals, bins=100, alpha=0.6, label='UNet imag', color='C1')
axes[1].axvline(-1.2, color='C0', ls=':', label='CCNF bounds (±1.2)')
axes[1].axvline(1.2, color='C0', ls=':')
axes[1].axvline(-1.0, color='C1', ls=':', label='UNet bounds (±1.0)')
axes[1].axvline(1.0, color='C1', ls=':')
axes[1].set_title('Imag Channel Distribution')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('value_distributions.png', bbox_inches='tight')
plt.show()

## 8. Forward Model Consistency Check

Run the full forward pass with physics constraints and compare
predicted vs measured diffraction. If CCNF loss is lower but patches
are blurrier, the smooth predictions may minimize Poisson NLL more
effectively (variance reduction at the cost of resolution).

In [ ]:
from ptycho_torch.model import PoissonLoss, MAELoss

physics_scale = batch[0]['physics_scaling_constant'].to(DEVICE)
probe_scaling = batch[2].to(DEVICE)
experiment_ids = batch[0]['experiment_id'].to(DEVICE)
modified_output_scale = torch.sqrt(1 / (probe_scaling**2 * physics_scale + 1e-9))

with torch.no_grad():
    ccnf_pred, ccnf_real, ccnf_imag = ccnf_model(
        x, positions, probe, in_scale, modified_output_scale,
        experiment_ids=experiment_ids,
    )
    unet_pred, unet_real, unet_imag = unet_model(
        x, positions, probe, in_scale, modified_output_scale,
        experiment_ids=experiment_ids,
    )

poisson_loss = PoissonLoss()
mae_loss = MAELoss()

ccnf_poisson = poisson_loss(ccnf_pred, x).mean().item()
unet_poisson = poisson_loss(unet_pred, x).mean().item()
ccnf_mae = mae_loss(ccnf_pred, x).mean().item()
unet_mae = mae_loss(unet_pred, x).mean().item()

print(f'Poisson Loss — CCNF: {ccnf_poisson:.4f}, UNet: {unet_poisson:.4f}')
print(f'MAE Loss    — CCNF: {ccnf_mae:.4f}, UNet: {unet_mae:.4f}')

# Visualize residuals for first patch
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
p_idx = 0

diff_in = x[p_idx, c_idx].cpu().numpy()
ccnf_diff = ccnf_pred[p_idx, c_idx].cpu().numpy()
unet_diff = unet_pred[p_idx, c_idx].cpu().numpy()

axes[0, 0].imshow(np.log1p(diff_in), cmap='viridis'); axes[0, 0].set_title('Measured')
axes[0, 1].imshow(np.log1p(np.clip(ccnf_diff, 0, None)), cmap='viridis'); axes[0, 1].set_title('CCNF Predicted')
axes[0, 2].imshow(np.log1p(np.clip(unet_diff, 0, None)), cmap='viridis'); axes[0, 2].set_title('UNet Predicted')

ccnf_res = np.abs(ccnf_diff - diff_in)
unet_res = np.abs(unet_diff - diff_in)
vmax = max(ccnf_res.max(), unet_res.max())
axes[1, 0].set_visible(False)
axes[1, 1].imshow(ccnf_res, cmap='hot', vmax=vmax); axes[1, 1].set_title(f'CCNF |residual| (mean={ccnf_res.mean():.2f})')
axes[1, 2].imshow(unet_res, cmap='hot', vmax=vmax); axes[1, 2].set_title(f'UNet |residual| (mean={unet_res.mean():.2f})')

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig('forward_model_residuals.png', bbox_inches='tight')
plt.show()

## 9. Step-by-Step Reassembly

Manually recreate the assembly loop for a few batches to visualize intermediate canvas states.
This isolates whether blurriness worsens during stitching.

In [ ]:
N_ASSEMBLY_BATCHES = 10  # number of batches to assemble

# Compute canvas size from global coordinates
coords_global_all = subset.mmap_ptycho['coords_global']
center_of_mass = coords_global_all.squeeze().mean(dim=0).to(DEVICE)

adjusted = coords_global_all.squeeze() - center_of_mass.cpu()
max_offset_x = int(np.ceil(adjusted[:, 0].abs().max().item()))
max_offset_y = int(np.ceil(adjusted[:, 1].abs().max().item()))
canvas_h = MIDDLE_TRIM + 2 * max_offset_y
canvas_w = MIDDLE_TRIM + 2 * max_offset_x
canvas_size = (canvas_h, canvas_w)

print(f'Canvas size: {canvas_size}')
print(f'Center of mass: {center_of_mass.cpu().numpy()}')

# Initialize canvases for both models
ccnf_canvas = torch.zeros(canvas_size, device=DEVICE, dtype=torch.complex64)
ccnf_weights = torch.zeros(canvas_size, device=DEVICE, dtype=torch.float32)
unet_canvas = torch.zeros(canvas_size, device=DEVICE, dtype=torch.complex64)
unet_weights = torch.zeros(canvas_size, device=DEVICE, dtype=torch.float32)

accumulator = VectorizedWeightedAccumulator()
N = dc.N
middle = MIDDLE_TRIM
center_start = N // 2 - middle // 2
center_end = N // 2 + middle // 2

with torch.no_grad():
    for i, batch in enumerate(loader):
        if i >= N_ASSEMBLY_BATCHES:
            break

        I_raw = batch[0]['images'].to(DEVICE)
        pos = batch[0]['coords_relative'].to(DEVICE)
        prb = batch[1].to(DEVICE)
        sc = batch[0]['rms_scaling_constant'].to(DEVICE)
        gcoords = batch[0]['coords_global'].to(DEVICE)

        # Predict from both models
        ccnf_raw = ccnf_model.forward_predict(I_raw, pos, prb, sc)
        unet_raw = unet_model.forward_predict(I_raw, pos, prb, sc)

        # Center crop
        prb_crop = prb[:, :, :, center_start:center_end, center_start:center_end]
        probe_mag_sq = torch.sum(torch.abs(prb_crop[0, 0, :, :, :]) ** 2, dim=0)

        for label, raw, canvas, weights in [
            ('ccnf', ccnf_raw, ccnf_canvas, ccnf_weights),
            ('unet', unet_raw, unet_canvas, unet_weights),
        ]:
            a = raw.real[:, :, center_start:center_end, center_start:center_end]
            b = raw.imag[:, :, center_start:center_end, center_start:center_end]
            B, C = a.shape[:2]

            O = torch.complex(a, b).view(B * C, middle, middle)
            gc2d = gcoords.squeeze(2).view(B * C, 2)
            rel = gc2d - center_of_mass.unsqueeze(0)
            canvas_center = torch.tensor(
                [canvas_size[1] // 2, canvas_size[0] // 2],
                device=DEVICE, dtype=torch.float32,
            )
            cpos = rel + canvas_center.unsqueeze(0)

            accumulator.accumulate_batch(
                canvas, weights, O, cpos, probe_mag_sq,
                patch_size=middle, uniform_weighting=False,
            )

        if (i + 1) % 5 == 0:
            print(f'  Assembled {i + 1} batches')

print(f'Assembled {min(i + 1, N_ASSEMBLY_BATCHES)} batches')

In [ ]:
# Visualize partially assembled canvases
ccnf_img = (ccnf_canvas / (ccnf_weights + 1e-12)).cpu()
unet_img = (unet_canvas / (unet_weights + 1e-12)).cpu()

w = 20  # border trim for display

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# CCNF
axes[0, 0].imshow(ccnf_img.real.numpy()[w:-w, w:-w], cmap='RdBu_r')
axes[0, 0].set_title('CCNF Real')
axes[0, 1].imshow(ccnf_img.imag.numpy()[w:-w, w:-w], cmap='RdBu_r')
axes[0, 1].set_title('CCNF Imag')
axes[0, 2].imshow(ccnf_img.abs().numpy()[w:-w, w:-w], cmap='gray')
axes[0, 2].set_title('CCNF |O|')
axes[0, 3].imshow(ccnf_img.angle().numpy()[w:-w, w:-w], cmap='twilight')
axes[0, 3].set_title('CCNF Phase')

# UNet
axes[1, 0].imshow(unet_img.real.numpy()[w:-w, w:-w], cmap='RdBu_r')
axes[1, 0].set_title('UNet Real')
axes[1, 1].imshow(unet_img.imag.numpy()[w:-w, w:-w], cmap='RdBu_r')
axes[1, 1].set_title('UNet Imag')
axes[1, 2].imshow(unet_img.abs().numpy()[w:-w, w:-w], cmap='gray')
axes[1, 2].set_title('UNet |O|')
axes[1, 3].imshow(unet_img.angle().numpy()[w:-w, w:-w], cmap='twilight')
axes[1, 3].set_title('UNet Phase')

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Partial Assembly ({min(i + 1, N_ASSEMBLY_BATCHES)} batches, no VarPro scaling)', fontsize=13)
plt.tight_layout()
plt.savefig('partial_assembly.png', bbox_inches='tight')
plt.show()

## 10. Full Reconstruction & VarPro Comparison

Run the full reconstruction pipeline for both models and compare
VarPro scaling constants (s1, s2) and final image quality.

In [ ]:
tc_infer = copy.deepcopy(tc)
tc_infer.device = DEVICE

# CCNF full reconstruction
print('Running CCNF full reconstruction...')
ccnf_result, ccnf_subset, ccnf_stats, ccnf_modified = reconstruct_image_barycentric(
    ccnf_model, dataset, tc_infer, dc, ccnf_mc, ic,
    gpu_ids=None, use_mixed_precision=True, verbose=True,
    return_diagnostics=True,
)
print(f'  s1={ccnf_stats[2]:.4f}, s2={ccnf_stats[3]:.4f}')
print(f'  Inference: {ccnf_stats[0]:.2f}s, Assembly: {ccnf_stats[1]:.2f}s')

In [ ]:
# UNet full reconstruction
print('Running UNet full reconstruction...')
unet_result, unet_subset, unet_stats, unet_modified = reconstruct_image_barycentric(
    unet_model, dataset, tc_infer, dc, unet_mc, ic,
    gpu_ids=None, use_mixed_precision=True, verbose=True,
    return_diagnostics=True,
)
print(f'  s1={unet_stats[2]:.4f}, s2={unet_stats[3]:.4f}')
print(f'  Inference: {unet_stats[0]:.2f}s, Assembly: {unet_stats[1]:.2f}s')

In [ ]:
# Compare final reconstructions
w = 20
cr = ccnf_result.cpu()
ur = unet_result.cpu()
if cr.dim() == 3:
    cr = cr[0]
if ur.dim() == 3:
    ur = ur[0]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

axes[0, 0].imshow(cr.real.numpy()[w:-w, w:-w], cmap='RdBu_r'); axes[0, 0].set_title('CCNF Real')
axes[0, 1].imshow(cr.imag.numpy()[w:-w, w:-w], cmap='RdBu_r'); axes[0, 1].set_title('CCNF Imag')
axes[0, 2].imshow(cr.abs().numpy()[w:-w, w:-w], cmap='gray'); axes[0, 2].set_title('CCNF |O|')
axes[0, 3].imshow(cr.angle().numpy()[w:-w, w:-w], cmap='twilight'); axes[0, 3].set_title('CCNF Phase')

axes[1, 0].imshow(ur.real.numpy()[w:-w, w:-w], cmap='RdBu_r'); axes[1, 0].set_title('UNet Real')
axes[1, 1].imshow(ur.imag.numpy()[w:-w, w:-w], cmap='RdBu_r'); axes[1, 1].set_title('UNet Imag')
axes[1, 2].imshow(ur.abs().numpy()[w:-w, w:-w], cmap='gray'); axes[1, 2].set_title('UNet |O|')
axes[1, 3].imshow(ur.angle().numpy()[w:-w, w:-w], cmap='twilight'); axes[1, 3].set_title('UNet Phase')

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Full Reconstruction — CCNF vs UNet (with VarPro scaling)', fontsize=13)
plt.tight_layout()
plt.savefig('full_reconstruction_comparison.png', bbox_inches='tight')
plt.show()

# Compare against ground truth if available
obj_list = ccnf_subset.data_dict.get('objectGuess', [])
if len(obj_list) > 0:
    gt = obj_list[min(EXPERIMENT_NUMBER, len(obj_list) - 1)]
    if hasattr(gt, 'squeeze'):
        gt = gt.squeeze()
    gt_amp = np.abs(gt)
    gt_phase = np.angle(gt)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(gt_amp[w:-w, w:-w], cmap='gray')
    axes[0].set_title('Ground Truth |O|')
    axes[1].imshow(gt_phase[w:-w, w:-w], cmap='twilight')
    axes[1].set_title('Ground Truth Phase')
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print('No ground truth available in dataset.')

## 11. Ablation — Uniform vs Probe Weighting

Re-run assembly with uniform weighting to check if probe-weighted
stitching interacts poorly with CCNF outputs.

In [ ]:
# Re-run CCNF with uniform weighting
ic_uniform = copy.deepcopy(ic)
ic_uniform.patch_weighting = 'uniform'

print('Running CCNF with uniform weighting...')
ccnf_uniform, _, ccnf_u_stats, _ = reconstruct_image_barycentric(
    ccnf_model, dataset, tc_infer, dc, ccnf_mc, ic_uniform,
    gpu_ids=None, use_mixed_precision=True, verbose=False,
    return_diagnostics=True,
)

cu = ccnf_uniform.cpu()
if cu.dim() == 3:
    cu = cu[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(cr.abs().numpy()[w:-w, w:-w], cmap='gray')
axes[0].set_title('CCNF Probe-weighted')
axes[1].imshow(cu.abs().numpy()[w:-w, w:-w], cmap='gray')
axes[1].set_title('CCNF Uniform-weighted')
diff = (cr.abs() - cu.abs()).numpy()[w:-w, w:-w]
axes[2].imshow(diff, cmap='RdBu_r', vmin=-diff.std()*3, vmax=diff.std()*3)
axes[2].set_title('Difference (probe - uniform)')

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.savefig('weighting_ablation.png', bbox_inches='tight')
plt.show()

## 12. Coordinate Consistency (CCNF-specific)

Check whether the coordinate distributions at inference match training.
The neural field decoder is conditioned on coordinates — if the
inference coordinate range exceeds training, the MLP extrapolates
and may produce blurry outputs.

In [ ]:
from ptycho_torch.beta_modules.latent_model import compute_canvas_scale

# Check coordinate distributions for inference data
coords_sample = batch[0]['coords_relative'].to(DEVICE)

M_lat = getattr(ccnf_mc, 'latent_canvas_size', 8)
scale = compute_canvas_scale(coords_sample, dc.N, M_lat)

print(f'=== Coordinate Analysis ===')
print(f'Latent canvas size: {M_lat}')
print(f'N (patch size): {dc.N}')
print(f'coords_relative shape: {coords_sample.shape}')

c2d = coords_sample.squeeze(2)  # (B, C, 2)
print(f'\nCoordinate ranges (this batch):')
print(f'  x: [{c2d[:,:,0].min():.2f}, {c2d[:,:,0].max():.2f}]')
print(f'  y: [{c2d[:,:,1].min():.2f}, {c2d[:,:,1].max():.2f}]')
print(f'  x span: {c2d[:,:,0].max() - c2d[:,:,0].min():.2f} px')
print(f'  y span: {c2d[:,:,1].max() - c2d[:,:,1].min():.2f} px')

print(f'\nCanvas scale (px/canvas_px):')
print(f'  Per-batch scale: {scale.squeeze().cpu().numpy()}')
print(f'  scale x range: [{scale[:,:,0].min():.4f}, {scale[:,:,0].max():.4f}]')
print(f'  scale y range: [{scale[:,:,1].min():.4f}, {scale[:,:,1].max():.4f}]')

# Normalized query coordinates (what the neural field decoder sees)
total_extent = (c2d.max(dim=1).values - c2d.min(dim=1).values + dc.N).clamp(min=float(dc.N))
print(f'\nTotal extent (bbox + N):')
print(f'  x: {total_extent[0, 0]:.1f}, y: {total_extent[0, 1]:.1f}')
print(f'\nRatio canvas_size/extent (should be consistent with training):')
print(f'  x: {(M_lat - 2) / total_extent[0, 0]:.4f}')
print(f'  y: {(M_lat - 2) / total_extent[0, 1]:.4f}')

In [ ]:
# Scatter plot of all scan positions for this experiment
all_coords = coords_global_all.squeeze().cpu().numpy()

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.scatter(all_coords[:, 0], all_coords[:, 1], s=2, alpha=0.3)
ax.set_xlabel('x (pixels)')
ax.set_ylabel('y (pixels)')
ax.set_title(f'Scan positions — Experiment {EXPERIMENT_NUMBER}')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Histogram of coordinate spacings
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Per-group coordinate spans (C patches within each group)
spans = []
for batch_i in loader:
    cr_batch = batch_i[0]['coords_relative'].squeeze(2)  # (B, C, 2)
    span = cr_batch.max(dim=1).values - cr_batch.min(dim=1).values  # (B, 2)
    spans.append(span)
    if len(spans) > 50:
        break
spans = torch.cat(spans, dim=0).numpy()

axes[0].hist(spans[:, 0], bins=50, alpha=0.7, label='x span')
axes[0].hist(spans[:, 1], bins=50, alpha=0.7, label='y span')
axes[0].set_xlabel('Coordinate span within group (pixels)')
axes[0].set_ylabel('Count')
axes[0].set_title('Per-group coordinate extent')
axes[0].legend()

# Scale factors
scales = []
for batch_i in loader:
    cr_batch = batch_i[0]['coords_relative'].to(DEVICE)
    sc = compute_canvas_scale(cr_batch, dc.N, M_lat)
    scales.append(sc.squeeze(1).cpu())
    if len(scales) > 50:
        break
scales = torch.cat(scales, dim=0).numpy()

axes[1].hist(scales[:, 0], bins=50, alpha=0.7, label='scale_x')
axes[1].hist(scales[:, 1], bins=50, alpha=0.7, label='scale_y')
axes[1].set_xlabel('Canvas scale (canvas_px / real_px)')
axes[1].set_ylabel('Count')
axes[1].set_title('Per-batch canvas scale distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('coordinate_analysis.png', bbox_inches='tight')
plt.show()

## Summary

**Interpretation guide:**

- **Cell 5** (patch visualization): If CCNF patches are individually blurry → spectral bias in neural field MLP
- **Cell 6** (power spectra): If CCNF power drops faster at high frequencies → confirms spectral bias
- **Cell 7** (value distributions): If values cluster near activation bounds → saturation problem
- **Cell 8** (forward model): If CCNF loss is lower despite blur → smooth predictions reduce Poisson variance
- **Cell 9-10** (assembly): If patches are sharp but assembly is blurry → stitching/VarPro issue
- **Cell 11** (weighting ablation): If uniform weighting helps → probe interaction problem
- **Cell 12** (coordinates): If scale distributions vary wildly → coordinate generalization issue